# 11_SageMaker_Deployment_Endpoint.ipynb

## Purpose

This notebook deploys the **latest eligible model version from the SageMaker Model Registry** created by Notebook 10.

Flow: `Notebook 10 Pipeline → Model Registry → Notebook 11 → Deployment bundle → SageMaker endpoint → Inference test`

- Notebook 10 must have completed successfully at least once.
- This notebook does **not** retrain the model.
- The stable endpoint name remains `heart-attack-team05-s502-endpoint`.
- The deployment is derived from the selected Model Registry version, not the old hard-coded Stage 10 artifact.

## 1. FIRST RUN — Imports and AWS configuration

In [1]:
from pathlib import Path
from datetime import datetime, timezone
from botocore.exceptions import ClientError
import boto3, json, os, shutil, tarfile, time
import pandas as pd
from sagemaker.core import image_uris

REGION = "ap-southeast-1"
ROLE_ARN = "arn:aws:iam::044528205969:role/SageMakerExecutionRole-ITI113-Team05"
MODEL_PACKAGE_GROUP = "iti113-team05-heart-attack-risk-models"
ENDPOINT_NAME = "heart-attack-team05-s502-endpoint"
INSTANCE_TYPE = "ml.m5.large"
INSTANCE_COUNT = 1
ALLOWED_APPROVAL_STATUSES = {"Approved", "PendingManualApproval"}
FORCE_REDEPLOY = False

os.environ["AWS_DEFAULT_REGION"] = REGION
os.environ["AWS_REGION"] = REGION
boto3.setup_default_session(region_name=REGION)
session = boto3.Session(region_name=REGION)
sm_client = boto3.client("sagemaker", region_name=REGION)
runtime = boto3.client("sagemaker-runtime", region_name=REGION)
s3_client = boto3.client("s3", region_name=REGION)
sts_client = boto3.client("sts", region_name=REGION)
ACCOUNT_ID = sts_client.get_caller_identity()["Account"]

print("Region              :", REGION)
print("Account             :", ACCOUNT_ID)
print("Role                :", ROLE_ARN)
print("Model package group :", MODEL_PACKAGE_GROUP)
print("Endpoint            :", ENDPOINT_NAME)

sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml


sagemaker.config INFO - Not applying SDK defaults from location: /home/sagemaker-user/.config/sagemaker/config.yaml


Region              : ap-southeast-1
Account             : 044528205969
Role                : arn:aws:iam::044528205969:role/SageMakerExecutionRole-ITI113-Team05
Model package group : iti113-team05-heart-attack-risk-models
Endpoint            : heart-attack-team05-s502-endpoint


## 2. FIRST RUN — Locate project files created/used by Notebook 10

In [2]:
EXPECTED_PROJECT_FOLDER = "Heart_Attack_Risk_Assessment"
current = Path.cwd().resolve()
if current.name == EXPECTED_PROJECT_FOLDER:
    PROJECT_ROOT = current
else:
    PROJECT_ROOT = next((p for p in current.parents if p.name == EXPECTED_PROJECT_FOLDER), None)
if PROJECT_ROOT is None:
    fallback = Path("/home/sagemaker-user/Heart_Attack_Risk_Assessment")
    if fallback.exists(): PROJECT_ROOT = fallback
    else: raise FileNotFoundError("Cannot locate Heart_Attack_Risk_Assessment.")

DATA_DIR = PROJECT_ROOT / "data"
ARTIFACT_DIR = PROJECT_ROOT / "artifacts"
PIPELINE10_DIR = ARTIFACT_DIR / "stage10_pipeline"
METADATA_TEMPLATE_FILE = PIPELINE10_DIR / "metadata_template.json"
TRAIN_RAW_FILE = DATA_DIR / "full_train_raw.csv"
DEPLOY_WORK_DIR = ARTIFACT_DIR / "stage11_deployment"
SOURCE_DIR = DEPLOY_WORK_DIR / "source"
SOURCE_DIR.mkdir(parents=True, exist_ok=True)

for name, path in {"metadata template": METADATA_TEMPLATE_FILE, "training-format data": TRAIN_RAW_FILE}.items():
    print(f"{'FOUND' if path.exists() else 'MISSING':7} | {name:24} | {path}")
    if not path.exists(): raise FileNotFoundError(str(path))
print("\n✅ Project files available.")

FOUND   | metadata template        | /home/sagemaker-user/Heart_Attack_Risk_Assessment/artifacts/stage10_pipeline/metadata_template.json
FOUND   | training-format data     | /home/sagemaker-user/Heart_Attack_Risk_Assessment/data/full_train_raw.csv

✅ Project files available.


## 3. FIRST RUN — Select the newest eligible model from Model Registry

The notebook accepts `Approved` or `PendingManualApproval` for the academic demo, but never `Rejected`. For a production-style governance rule, change `ALLOWED_APPROVAL_STATUSES` to `{"Approved"}`.

In [3]:
print("=" * 80); print("MODEL REGISTRY SELECTION"); print("=" * 80)
response = sm_client.list_model_packages(ModelPackageGroupName=MODEL_PACKAGE_GROUP, ModelPackageType="Versioned", SortBy="CreationTime", SortOrder="Descending", MaxResults=50)
summaries = response.get("ModelPackageSummaryList", [])
if not summaries: raise RuntimeError("No model versions found. Run Notebook 10 first.")
selected_desc = None
for summary in summaries:
    desc = sm_client.describe_model_package(ModelPackageName=summary["ModelPackageArn"], IncludedData="MetadataOnly")
    if desc.get("ModelPackageStatus") == "Completed" and desc.get("ModelApprovalStatus") in ALLOWED_APPROVAL_STATUSES:
        selected_desc = desc; break
if selected_desc is None: raise RuntimeError("No eligible completed model package found.")
MODEL_PACKAGE_ARN = selected_desc["ModelPackageArn"]
MODEL_PACKAGE_VERSION = selected_desc["ModelPackageVersion"]
MODEL_APPROVAL_STATUS = selected_desc["ModelApprovalStatus"]
MODEL_PACKAGE_STATUS = selected_desc["ModelPackageStatus"]
print("Model package :", MODEL_PACKAGE_ARN)
print("Version       :", MODEL_PACKAGE_VERSION)
print("Status        :", MODEL_PACKAGE_STATUS)
print("Approval      :", MODEL_APPROVAL_STATUS)
if MODEL_APPROVAL_STATUS == "PendingManualApproval":
    print("\n⚠️ Academic demo: selected model is PendingManualApproval. It passed the Notebook 10 quality gate but is not manually Approved.")
if MODEL_APPROVAL_STATUS == "Rejected": raise RuntimeError("Refusing to deploy a Rejected model.")
print("\n✅ Eligible registry model selected.")

MODEL REGISTRY SELECTION
Model package : arn:aws:sagemaker:ap-southeast-1:044528205969:model-package/iti113-team05-heart-attack-risk-models/2
Version       : 2
Status        : Completed
Approval      : PendingManualApproval

⚠️ Academic demo: selected model is PendingManualApproval. It passed the Notebook 10 quality gate but is not manually Approved.

✅ Eligible registry model selected.


## 4. FIRST RUN — Resolve the registered model artifact

In [4]:
inference_spec = selected_desc.get("InferenceSpecification", {})
containers = inference_spec.get("Containers", [])
if not containers: raise RuntimeError("Selected package has no inference container.")
registry_container = containers[0]
REGISTRY_IMAGE_URI = registry_container.get("Image")
REGISTRY_MODEL_DATA = registry_container.get("ModelDataUrl")
if not REGISTRY_MODEL_DATA:
    REGISTRY_MODEL_DATA = registry_container.get("ModelDataSource", {}).get("S3DataSource", {}).get("S3Uri")
if not REGISTRY_MODEL_DATA: raise RuntimeError("Could not resolve registered model artifact S3 URI.")
print("Registry image         :", REGISTRY_IMAGE_URI)
print("Registry model artifact:", REGISTRY_MODEL_DATA)
print("✅ Registry artifact resolved.")

Registry image         : 121021644041.dkr.ecr.ap-southeast-1.amazonaws.com/sagemaker-scikit-learn:1.4-2-py312-cpu-py3
Registry model artifact: s3://sagemaker-ap-southeast-1-044528205969/team05-heart-xgboost-train/pipelines-xldnxhku9k3r-TrainHeartAttackXGBo-tJcqetwCrs/output/model.tar.gz
✅ Registry artifact resolved.


## 5. FIRST RUN — Download and verify the registered model artifact

In [5]:
def split_s3_uri(uri):
    if not uri.startswith("s3://"): raise ValueError(uri)
    bucket_key = uri[5:]; return bucket_key.split("/", 1)
registry_bucket, registry_key = split_s3_uri(REGISTRY_MODEL_DATA)
REGISTRY_MODEL_TAR = DEPLOY_WORK_DIR / "registry_model.tar.gz"
s3_client.download_file(registry_bucket, registry_key, str(REGISTRY_MODEL_TAR))
with tarfile.open(REGISTRY_MODEL_TAR, "r:gz") as archive:
    member_names = [m.name for m in archive.getmembers() if m.isfile()]
print("Downloaded:", REGISTRY_MODEL_TAR)
print("Files in registered artifact:")
for name in member_names: print(" -", name)
if not any(Path(n).name == "model.joblib" for n in member_names): raise RuntimeError("Registered artifact does not contain model.joblib.")
print("✅ Registered artifact contains model.joblib.")

Downloaded: /home/sagemaker-user/Heart_Attack_Risk_Assessment/artifacts/stage11_deployment/registry_model.tar.gz
Files in registered artifact:
 - model.joblib
 - training_metadata.json
✅ Registered artifact contains model.joblib.


## 6. FIRST RUN — Build a deployment artifact from the registered model

This preserves the registered trained model and adds the inference metadata generated by Notebook 10. No retraining occurs.

In [6]:
EXTRACT_DIR = DEPLOY_WORK_DIR / "registry_model_extracted"
if EXTRACT_DIR.exists(): shutil.rmtree(EXTRACT_DIR)
EXTRACT_DIR.mkdir(parents=True, exist_ok=True)

def safe_extract_tar(tar_path, destination):
    destination = Path(destination).resolve()
    with tarfile.open(tar_path, "r:gz") as archive:
        for member in archive.getmembers():
            target = (destination / member.name).resolve()
            if destination != target and destination not in target.parents:
                raise RuntimeError(f"Unsafe archive path: {member.name}")
        archive.extractall(destination)

safe_extract_tar(REGISTRY_MODEL_TAR, EXTRACT_DIR)
with open(METADATA_TEMPLATE_FILE, "r", encoding="utf-8") as f: deployment_metadata = json.load(f)
deployment_metadata.update({
    "model_package_group": MODEL_PACKAGE_GROUP,
    "model_package_arn": MODEL_PACKAGE_ARN,
    "model_package_version": MODEL_PACKAGE_VERSION,
    "model_approval_status": MODEL_APPROVAL_STATUS,
    "registry_model_data": REGISTRY_MODEL_DATA,
    "deployment_notebook": "11_SageMaker_Deployment_Endpoint",
})
DEPLOYMENT_METADATA_FILE = EXTRACT_DIR / "metadata.json"
DEPLOYMENT_METADATA_FILE.write_text(json.dumps(deployment_metadata, indent=2), encoding="utf-8")
model_matches = list(EXTRACT_DIR.rglob("model.joblib"))
if not model_matches: raise FileNotFoundError("model.joblib not found after extraction.")
MODEL_JOBLIB_ROOT = EXTRACT_DIR / "model.joblib"
if model_matches[0].resolve() != MODEL_JOBLIB_ROOT.resolve(): shutil.copy2(model_matches[0], MODEL_JOBLIB_ROOT)
DEPLOYMENT_MODEL_TAR = DEPLOY_WORK_DIR / "model.tar.gz"
with tarfile.open(DEPLOYMENT_MODEL_TAR, "w:gz") as archive:
    archive.add(MODEL_JOBLIB_ROOT, arcname="model.joblib")
    archive.add(DEPLOYMENT_METADATA_FILE, arcname="metadata.json")
print("Deployment artifact:", DEPLOYMENT_MODEL_TAR)
print("Registry version   :", MODEL_PACKAGE_VERSION)
print("Frozen threshold   :", deployment_metadata["threshold"])
print("Feature count      :", len(deployment_metadata["features"]))
print("✅ Registry-derived deployment artifact created.")

Deployment artifact: /home/sagemaker-user/Heart_Attack_Risk_Assessment/artifacts/stage11_deployment/model.tar.gz
Registry version   : 2
Frozen threshold   : 0.52
Feature count      : 39
✅ Registry-derived deployment artifact created.


/tmp/ipykernel_377/574382672.py:12: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  archive.extractall(destination)


## 7. FIRST RUN — Upload the registry-derived deployment artifact to S3

In [7]:
DEPLOY_BUCKET = f"sagemaker-{REGION}-{ACCOUNT_ID}"
DEPLOYMENT_PREFIX = f"heart-attack-risk/deployment11/model-package-v{MODEL_PACKAGE_VERSION}"
DEPLOYMENT_MODEL_KEY = f"{DEPLOYMENT_PREFIX}/model.tar.gz"
DEPLOYMENT_MODEL_S3_URI = f"s3://{DEPLOY_BUCKET}/{DEPLOYMENT_MODEL_KEY}"
s3_client.upload_file(str(DEPLOYMENT_MODEL_TAR), DEPLOY_BUCKET, DEPLOYMENT_MODEL_KEY)
print("Uploaded:", DEPLOYMENT_MODEL_S3_URI)
print("✅ Deployment artifact uploaded.")

Uploaded: s3://sagemaker-ap-southeast-1-044528205969/heart-attack-risk/deployment11/model-package-v2/model.tar.gz
✅ Deployment artifact uploaded.


## 8. FIRST RUN — Create `inference.py` and serving requirements

The inference container keeps its built-in scikit-learn 1.4.2. XGBoost 3.4.1 is installed because the trained pipeline contains `XGBClassifier`.

In [8]:
INFERENCE_PY = SOURCE_DIR / "inference.py"
REQUIREMENTS_TXT = SOURCE_DIR / "requirements.txt"
INFERENCE_PY.write_text('import json\nimport os\nimport joblib\nimport numpy as np\nimport pandas as pd\n\ndef _load_metadata(model_dir):\n    with open(os.path.join(model_dir, "metadata.json"), "r", encoding="utf-8") as f:\n        return json.load(f)\n\ndef model_fn(model_dir):\n    pipeline = joblib.load(os.path.join(model_dir, "model.joblib"))\n    return {"pipeline": pipeline, "metadata": _load_metadata(model_dir)}\n\ndef input_fn(request_body, request_content_type):\n    if request_content_type != "application/json":\n        raise ValueError(f"Unsupported Content-Type: {request_content_type}. Use application/json.")\n    payload = json.loads(request_body)\n    if isinstance(payload, (dict, list)): return payload\n    raise ValueError("JSON body must be an object or a list of objects.")\n\ndef predict_fn(input_data, model):\n    pipeline = model["pipeline"]; metadata = model["metadata"]\n    features = metadata["features"]; threshold = float(metadata["threshold"])\n    if isinstance(input_data, dict): records, single = [input_data], True\n    elif isinstance(input_data, list): records, single = input_data, False\n    else: raise ValueError("Invalid parsed input.")\n    if not records: raise ValueError("At least one record is required.")\n    expected = set(features); rows = []\n    for record in records:\n        if not isinstance(record, dict): raise ValueError("Each input record must be a JSON object.")\n        unknown = sorted(set(record.keys()) - expected)\n        if unknown: raise ValueError(f"Unexpected input fields: {unknown}")\n        rows.append({feature: record.get(feature, np.nan) for feature in features})\n    frame = pd.DataFrame(rows, columns=features)\n    scores = pipeline.predict_proba(frame)[:, 1]\n    outputs = []\n    for score in scores:\n        score = float(score); positive = bool(score >= threshold)\n        outputs.append({\n            "association_score": round(score, 6),\n            "threshold": threshold,\n            "positive_class": positive,\n            "classification": (\n                "More similar to respondents who reported having had a heart attack"\n                if positive else\n                "More similar to respondents who reported not having had a heart attack"\n            ),\n            "model_package_version": metadata.get("model_package_version"),\n            "disclaimer": metadata["disclaimer"],\n        })\n    return outputs[0] if single else outputs\n\ndef output_fn(prediction, accept):\n    if accept not in ("application/json", "*/*"):\n        raise ValueError(f"Unsupported Accept type: {accept}")\n    return json.dumps(prediction), "application/json"\n', encoding="utf-8")
REQUIREMENTS_TXT.write_text("xgboost==3.4.1\njoblib\n", encoding="utf-8")
print("Inference script:", INFERENCE_PY)
print("Requirements:\n" + REQUIREMENTS_TXT.read_text())
print("✅ Serving source prepared.")

Inference script: /home/sagemaker-user/Heart_Attack_Risk_Assessment/artifacts/stage11_deployment/source/inference.py
Requirements:
xgboost==3.4.1
joblib

✅ Serving source prepared.


## 9. FIRST RUN — Package and upload inference source

In [9]:
SOURCE_ARCHIVE = DEPLOY_WORK_DIR / "sourcedir.tar.gz"
with tarfile.open(SOURCE_ARCHIVE, "w:gz") as archive:
    archive.add(INFERENCE_PY, arcname="inference.py")
    archive.add(REQUIREMENTS_TXT, arcname="requirements.txt")
SOURCE_CODE_KEY = f"{DEPLOYMENT_PREFIX}/source/sourcedir.tar.gz"
SOURCE_CODE_S3_URI = f"s3://{DEPLOY_BUCKET}/{SOURCE_CODE_KEY}"
s3_client.upload_file(str(SOURCE_ARCHIVE), DEPLOY_BUCKET, SOURCE_CODE_KEY)
print("Source S3 URI:", SOURCE_CODE_S3_URI)
print("✅ Inference source uploaded.")

Source S3 URI: s3://sagemaker-ap-southeast-1-044528205969/heart-attack-risk/deployment11/model-package-v2/source/sourcedir.tar.gz
✅ Inference source uploaded.


## 10. FIRST RUN — Resolve the explicit SageMaker Scikit-learn inference image

In [10]:
SKLEARN_CONTAINER_VERSION = "1.4-2-py312"
SKLEARN_IMAGE_URI = image_uris.retrieve(framework="sklearn", region=REGION, version=SKLEARN_CONTAINER_VERSION, image_scope="inference", instance_type=INSTANCE_TYPE)
print("Inference image:", SKLEARN_IMAGE_URI)
print("Registry image :", REGISTRY_IMAGE_URI)
if not SKLEARN_IMAGE_URI: raise RuntimeError("Could not resolve inference image.")
print("✅ Explicit inference image resolved.")

[08/18/26 02:54:14] INFO     Defaulting to only available Python version: py3                     ]8;id=2082892;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/image_uris.py\image_uris.py]8;;\:]8;id=2082893;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/image_uris.py#615\615]8;;\

Inference image: 121021644041.dkr.ecr.ap-southeast-1.amazonaws.com/sagemaker-scikit-learn:1.4-2-py312-cpu-py3
Registry image : 121021644041.dkr.ecr.ap-southeast-1.amazonaws.com/sagemaker-scikit-learn:1.4-2-py312-cpu-py3
✅ Explicit inference image resolved.


## 11. FIRST RUN — Build a sample request from training-format data

In [11]:
features = deployment_metadata["features"]
sample_df = pd.read_csv(TRAIN_RAW_FILE, nrows=1)
missing = [f for f in features if f not in sample_df.columns]
if missing: raise RuntimeError(f"Missing expected features: {missing}")
sample_row = sample_df[features].iloc[0]
SAMPLE_INPUT = {}
for key, value in sample_row.items():
    if pd.isna(value): SAMPLE_INPUT[key] = None
    elif hasattr(value, "item"): SAMPLE_INPUT[key] = value.item()
    else: SAMPLE_INPUT[key] = value
print(json.dumps(SAMPLE_INPUT, indent=2, default=str))
assert set(SAMPLE_INPUT) == set(features)
print("\n✅ SAMPLE_INPUT matches the model feature contract.")

{
  "State": "Texas",
  "Sex": "Female",
  "GeneralHealth": "Fair",
  "PhysicalHealthDays": 15.0,
  "MentalHealthDays": 30.0,
  "LastCheckupTime": "Within past year (anytime less than 12 months ago)",
  "PhysicalActivities": "Yes",
  "SleepHours": 4.0,
  "RemovedTeeth": "6 or more, but not all",
  "HadAngina": "No",
  "HadStroke": "No",
  "HadAsthma": "No",
  "HadSkinCancer": "No",
  "HadCOPD": "Yes",
  "HadDepressiveDisorder": "Yes",
  "HadKidneyDisease": "No",
  "HadArthritis": "Yes",
  "HadDiabetes": "No",
  "DeafOrHardOfHearing": "No",
  "BlindOrVisionDifficulty": "Yes",
  "DifficultyConcentrating": "No",
  "DifficultyWalking": "Yes",
  "DifficultyDressingBathing": "No",
  "DifficultyErrands": "No",
  "SmokerStatus": "Current smoker - now smokes every day",
  "ECigaretteUsage": "Never used e-cigarettes in my entire life",
  "ChestScan": "No",
  "RaceEthnicityCategory": "White only, Non-Hispanic",
  "AgeCategory": "Age 65 to 69",
  "HeightInMeters": 1.68,
  "WeightInKilograms": 48.5

# Phase B — Deploy or reuse the stable endpoint

## 12. FIRST RUN — Check whether the endpoint already exists

In [12]:
ENDPOINT_EXISTS = False; ENDPOINT_STATUS = None; EXISTING_ENDPOINT_CONFIG = None
try:
    endpoint_desc = sm_client.describe_endpoint(EndpointName=ENDPOINT_NAME)
    ENDPOINT_EXISTS = True
    ENDPOINT_STATUS = endpoint_desc["EndpointStatus"]
    EXISTING_ENDPOINT_CONFIG = endpoint_desc["EndpointConfigName"]
    print("Existing endpoint:", ENDPOINT_NAME); print("Status:", ENDPOINT_STATUS)
except ClientError as e:
    if e.response.get("Error", {}).get("Code") in ("ValidationException", "ResourceNotFound"):
        print("No existing endpoint found.")
    else: raise
print("FORCE_REDEPLOY:", FORCE_REDEPLOY)

No existing endpoint found.
FORCE_REDEPLOY: False


## 13. FIRST RUN — Create the SageMaker model and endpoint

If the endpoint is already `InService` and `FORCE_REDEPLOY=False`, it is reused. A missing/failed endpoint is created from the registry-derived model.

In [13]:
def wait_for_endpoint_deletion(endpoint_name, timeout_seconds=900):
    start=time.time()
    while True:
        try:
            desc=sm_client.describe_endpoint(EndpointName=endpoint_name)
            print("Waiting for deletion:", desc["EndpointStatus"])
        except ClientError: return
        if time.time()-start > timeout_seconds: raise TimeoutError("Timed out waiting for endpoint deletion.")
        time.sleep(10)

def delete_endpoint_only(endpoint_name):
    try:
        sm_client.delete_endpoint(EndpointName=endpoint_name)
        wait_for_endpoint_deletion(endpoint_name)
        print("✅ Endpoint deleted:", endpoint_name)
    except ClientError: pass

if ENDPOINT_EXISTS and ENDPOINT_STATUS == "InService" and not FORCE_REDEPLOY:
    print("✅ Reusing existing InService endpoint.")
    ACTIVE_MODEL_RESOURCE_NAME = None
    ACTIVE_ENDPOINT_CONFIG_NAME = EXISTING_ENDPOINT_CONFIG
else:
    if ENDPOINT_EXISTS:
        print("Removing existing endpoint before redeployment...")
        delete_endpoint_only(ENDPOINT_NAME)
    timestamp = datetime.now(timezone.utc).strftime("%Y%m%d-%H%M%S")
    ACTIVE_MODEL_RESOURCE_NAME = f"heart-attack-team05-s502-v{MODEL_PACKAGE_VERSION}-{timestamp}"
    ACTIVE_ENDPOINT_CONFIG_NAME = f"heart-attack-team05-s502-config-v{MODEL_PACKAGE_VERSION}-{timestamp}"
    print("="*80); print("CREATE SAGEMAKER MODEL"); print("="*80)
    model_response = sm_client.create_model(
        ModelName=ACTIVE_MODEL_RESOURCE_NAME,
        ExecutionRoleArn=ROLE_ARN,
        PrimaryContainer={
            "Image": SKLEARN_IMAGE_URI,
            "ModelDataUrl": DEPLOYMENT_MODEL_S3_URI,
            "Environment": {
                "SAGEMAKER_PROGRAM": "inference.py",
                "SAGEMAKER_SUBMIT_DIRECTORY": SOURCE_CODE_S3_URI,
                "SAGEMAKER_CONTAINER_LOG_LEVEL": "20",
                "SAGEMAKER_REGION": REGION,
            },
        },
        Tags=[
            {"Key":"Project","Value":"Heart_Attack_Risk_Assessment"},
            {"Key":"ModelPackageVersion","Value":str(MODEL_PACKAGE_VERSION)},
            {"Key":"Source","Value":"SageMakerModelRegistry"},
        ],
    )
    print("✅ SageMaker model created:", model_response["ModelArn"])
    model_desc = sm_client.describe_model(ModelName=ACTIVE_MODEL_RESOURCE_NAME)
    if model_desc["PrimaryContainer"].get("ModelDataUrl") != DEPLOYMENT_MODEL_S3_URI:
        raise RuntimeError("Wrong model artifact attached.")
    print("✅ Correct registry-derived artifact attached.")
    config_response = sm_client.create_endpoint_config(
        EndpointConfigName=ACTIVE_ENDPOINT_CONFIG_NAME,
        ProductionVariants=[{
            "VariantName":"AllTraffic",
            "ModelName":ACTIVE_MODEL_RESOURCE_NAME,
            "InitialInstanceCount":INSTANCE_COUNT,
            "InstanceType":INSTANCE_TYPE,
            "InitialVariantWeight":1.0,
        }],
        Tags=[{"Key":"ModelPackageVersion","Value":str(MODEL_PACKAGE_VERSION)}],
    )
    print("✅ Endpoint config created:", config_response["EndpointConfigArn"])
    endpoint_response = sm_client.create_endpoint(
        EndpointName=ENDPOINT_NAME,
        EndpointConfigName=ACTIVE_ENDPOINT_CONFIG_NAME,
        Tags=[{"Key":"ModelPackageVersion","Value":str(MODEL_PACKAGE_VERSION)}],
    )
    print("✅ Endpoint creation started:", endpoint_response["EndpointArn"])

CREATE SAGEMAKER MODEL


✅ SageMaker model created: arn:aws:sagemaker:ap-southeast-1:044528205969:model/heart-attack-team05-s502-v2-20260818-025906
✅ Correct registry-derived artifact attached.


✅ Endpoint config created: arn:aws:sagemaker:ap-southeast-1:044528205969:endpoint-config/heart-attack-team05-s502-config-v2-20260818-025906


✅ Endpoint creation started: arn:aws:sagemaker:ap-southeast-1:044528205969:endpoint/heart-attack-team05-s502-endpoint


## 14. FIRST RUN — Wait for endpoint to become `InService`

In [14]:
print("="*80); print("ENDPOINT STATUS"); print("="*80)
while True:
    endpoint_desc = sm_client.describe_endpoint(EndpointName=ENDPOINT_NAME)
    ENDPOINT_STATUS = endpoint_desc["EndpointStatus"]
    print("Endpoint status:", ENDPOINT_STATUS)
    if ENDPOINT_STATUS == "InService": print("\n✅ ENDPOINT IS INSERVICE"); break
    if ENDPOINT_STATUS == "Failed": raise RuntimeError("Endpoint deployment failed.\nReason: " + endpoint_desc.get("FailureReason", "Unknown failure"))
    time.sleep(30)
print("Endpoint:", ENDPOINT_NAME)
print("Endpoint config:", endpoint_desc["EndpointConfigName"])
print("Registry model version:", MODEL_PACKAGE_VERSION)

ENDPOINT STATUS
Endpoint status: Creating


Endpoint status: Creating


Endpoint status: Creating


Endpoint status: Creating


Endpoint status: InService

✅ ENDPOINT IS INSERVICE
Endpoint: heart-attack-team05-s502-endpoint
Endpoint config: heart-attack-team05-s502-config-v2-20260818-025906
Registry model version: 2


# Phase C — Invoke and reuse the endpoint

## 15. FIRST RUN — Invoke the real endpoint

In [15]:
response = runtime.invoke_endpoint(EndpointName=ENDPOINT_NAME, ContentType="application/json", Accept="application/json", Body=json.dumps(SAMPLE_INPUT).encode("utf-8"))
body = response["Body"].read().decode("utf-8")
try: result=json.loads(body)
except json.JSONDecodeError: result=body
print(json.dumps(result, indent=2) if isinstance(result,(dict,list)) else result)

{
  "association_score": 0.745162,
  "threshold": 0.52,
  "positive_class": true,
  "classification": "More similar to respondents who reported having had a heart attack",
  "model_package_version": 2,
  "disclaimer": "Educational prototype only. This output describes similarity to respondents who reported a previous heart attack. It does not predict a future heart attack and is not a medical diagnosis."
}


## 16. OPTIONAL — Reusable prediction helper

In [16]:
def invoke_heart_attack_endpoint(record):
    response=runtime.invoke_endpoint(EndpointName=ENDPOINT_NAME, ContentType="application/json", Accept="application/json", Body=json.dumps(record).encode("utf-8"))
    body=response["Body"].read().decode("utf-8")
    try: return json.loads(body)
    except json.JSONDecodeError: return body
print("✅ invoke_heart_attack_endpoint(record) ready.")

✅ invoke_heart_attack_endpoint(record) ready.


## 17. FIRST RUN — Save deployment evidence and registry provenance

In [ ]:
endpoint_desc=sm_client.describe_endpoint(EndpointName=ENDPOINT_NAME)
deployment_evidence={
    "timestamp_utc":datetime.now(timezone.utc).isoformat(),
    "region":REGION,
    "endpoint_name":ENDPOINT_NAME,
    "endpoint_status":endpoint_desc["EndpointStatus"],
    "endpoint_config_name":endpoint_desc["EndpointConfigName"],
    "instance_type":INSTANCE_TYPE,
    "model_package_group":MODEL_PACKAGE_GROUP,
    "model_package_arn":MODEL_PACKAGE_ARN,
    "model_package_version":MODEL_PACKAGE_VERSION,
    "model_approval_status":MODEL_APPROVAL_STATUS,
    "registry_model_data":REGISTRY_MODEL_DATA,
    "deployment_model_data":DEPLOYMENT_MODEL_S3_URI,
    "inference_image":SKLEARN_IMAGE_URI,
    "threshold":deployment_metadata["threshold"],
    "feature_count":len(deployment_metadata["features"]),
    "scope":"controlled_academic_demonstration",
}
DEPLOYMENT_EVIDENCE_FILE=DEPLOY_WORK_DIR/"deployment_evidence.json"
DEPLOYMENT_EVIDENCE_FILE.write_text(json.dumps(deployment_evidence, indent=2, default=str), encoding="utf-8")
print(json.dumps(deployment_evidence, indent=2, default=str))
print("Saved:", DEPLOYMENT_EVIDENCE_FILE)

# Phase D — Manual cleanup

## 18. 🛑 DO NOT RUN unless you intentionally want to delete the hosted endpoint

This removes the endpoint, its endpoint configuration, and its SageMaker model resource. It does **not** delete Notebook 10, pipeline history, Model Registry versions, or registered training artifacts.

In [ ]:
CONFIRM_DELETE = True
if not CONFIRM_DELETE:
    print("Cleanup skipped. Set CONFIRM_DELETE = True only when you intentionally want to remove the hosted endpoint.")
else:
    endpoint_desc=sm_client.describe_endpoint(EndpointName=ENDPOINT_NAME)
    endpoint_config_name=endpoint_desc["EndpointConfigName"]
    config_desc=sm_client.describe_endpoint_config(EndpointConfigName=endpoint_config_name)
    model_names=[v["ModelName"] for v in config_desc["ProductionVariants"]]
    sm_client.delete_endpoint(EndpointName=ENDPOINT_NAME)
    wait_for_endpoint_deletion(ENDPOINT_NAME)
    try: sm_client.delete_endpoint_config(EndpointConfigName=endpoint_config_name)
    except ClientError as e: print(e)
    for model_name in model_names:
        try: sm_client.delete_model(ModelName=model_name)
        except ClientError as e: print(e)
    print("✅ Hosted resources cleaned up. Registry and Notebook 10 resources remain intact.")

In [ ]:
# ============================================================
# STANDALONE — DELETE CURRENT SAGEMAKER ENDPOINT
#
# Safe after a kernel restart.
#
# Deletes:
#   1. SageMaker real-time endpoint
#   2. Endpoint configuration
#   3. SageMaker model resource(s) attached to endpoint
#
# Does NOT delete:
#   - SageMaker Pipeline
#   - Pipeline executions
#   - Model Registry / registered model versions
#   - Training jobs
#   - Processing jobs
#   - S3 model artifacts
# ============================================================

import boto3
import time
from botocore.exceptions import ClientError


# ============================================================
# CONFIGURATION
# ============================================================

REGION = "ap-southeast-1"

ENDPOINT_NAME = (
    "heart-attack-team05-s502-endpoint"
)


# ============================================================
# CREATE SAGEMAKER CLIENT
# ============================================================

sm_client = boto3.client(
    "sagemaker",
    region_name=REGION
)


print("=" * 80)
print("SAGEMAKER ENDPOINT CLEANUP")
print("=" * 80)

print("Region   :", REGION)
print("Endpoint :", ENDPOINT_NAME)


# ============================================================
# STEP 1 — FIND ENDPOINT
# ============================================================

try:

    endpoint_desc = (
        sm_client.describe_endpoint(
            EndpointName=ENDPOINT_NAME
        )
    )

except ClientError as e:

    error_code = (
        e.response
        .get("Error", {})
        .get("Code")
    )

    if error_code in (
        "ValidationException",
        "ResourceNotFound",
    ):

        print(
            "\n✅ Endpoint does not exist."
        )

        print(
            "Nothing needs to be deleted."
        )

        endpoint_desc = None

    else:
        raise


# ============================================================
# CONTINUE ONLY IF ENDPOINT EXISTS
# ============================================================

if endpoint_desc is not None:

    endpoint_status = (
        endpoint_desc[
            "EndpointStatus"
        ]
    )

    endpoint_config_name = (
        endpoint_desc[
            "EndpointConfigName"
        ]
    )


    print(
        "\nEndpoint status:"
    )

    print(
        endpoint_status
    )


    print(
        "\nEndpoint configuration:"
    )

    print(
        endpoint_config_name
    )


    # ========================================================
    # STEP 2 — FIND MODEL RESOURCE(S)
    #
    # We must do this BEFORE deleting endpoint config.
    # ========================================================

    config_desc = (
        sm_client.describe_endpoint_config(
            EndpointConfigName=
                endpoint_config_name
        )
    )


    model_names = [

        variant["ModelName"]

        for variant
        in config_desc[
            "ProductionVariants"
        ]

    ]


    print(
        "\nAttached SageMaker model resource(s):"
    )


    for model_name in model_names:

        print(
            " -",
            model_name
        )


    # ========================================================
    # STEP 3 — DELETE ENDPOINT
    # ========================================================

    print("\n" + "=" * 80)

    print(
        "STEP 1 — DELETE ENDPOINT"
    )

    print("=" * 80)


    sm_client.delete_endpoint(
        EndpointName=
            ENDPOINT_NAME
    )


    print(
        "Endpoint deletion requested."
    )


    # ========================================================
    # STEP 4 — WAIT FOR ENDPOINT TO DISAPPEAR
    # ========================================================

    while True:

        try:

            current_desc = (
                sm_client.describe_endpoint(
                    EndpointName=
                        ENDPOINT_NAME
                )
            )

            current_status = (
                current_desc[
                    "EndpointStatus"
                ]
            )


            print(
                "Current status:",
                current_status
            )


            time.sleep(10)


        except ClientError as e:

            error_code = (
                e.response
                .get("Error", {})
                .get("Code")
            )


            if error_code in (
                "ValidationException",
                "ResourceNotFound",
            ):

                print(
                    "\n✅ Endpoint deleted."
                )

                break


            raise


    # ========================================================
    # STEP 5 — DELETE ENDPOINT CONFIGURATION
    # ========================================================

    print("\n" + "=" * 80)

    print(
        "STEP 2 — DELETE ENDPOINT CONFIGURATION"
    )

    print("=" * 80)


    try:

        sm_client.delete_endpoint_config(
            EndpointConfigName=
                endpoint_config_name
        )


        print(
            "✅ Endpoint configuration deleted:"
        )

        print(
            endpoint_config_name
        )


    except ClientError as e:

        print(
            "⚠️ Could not delete endpoint configuration."
        )

        print(e)


    # ========================================================
    # STEP 6 — DELETE SAGEMAKER MODEL RESOURCE(S)
    # ========================================================

    print("\n" + "=" * 80)

    print(
        "STEP 3 — DELETE SAGEMAKER MODEL RESOURCE(S)"
    )

    print("=" * 80)


    for model_name in model_names:

        try:

            sm_client.delete_model(
                ModelName=
                    model_name
            )


            print(
                "✅ Deleted model:"
            )

            print(
                model_name
            )


        except ClientError as e:

            print(
                "⚠️ Could not delete model:"
            )

            print(
                model_name
            )

            print(e)


    # ========================================================
    # FINAL VERIFICATION
    # ========================================================

    print("\n" + "=" * 80)

    print(
        "FINAL VERIFICATION"
    )

    print("=" * 80)


    try:

        sm_client.describe_endpoint(
            EndpointName=
                ENDPOINT_NAME
        )

        print(
            "❌ Endpoint still exists."
        )


    except ClientError:

        print(
            "✅ Endpoint no longer exists."
        )


    print("\n" + "=" * 80)

    print(
        "✅ ENDPOINT CLEANUP COMPLETED"
    )

    print("=" * 80)


    print(
        "\nPreserved:"
    )

    print(
        "✅ Notebook 10 pipeline"
    )

    print(
        "✅ Pipeline execution history"
    )

    print(
        "✅ Model Registry"
    )

    print(
        "✅ Registered model versions"
    )

    print(
        "✅ S3 model artifacts"
    )

# Quick Run Guide

For a **fresh deployment after deleting the old endpoint**:

1. Run Sections **1 → 11**.
2. Section 3 selects the newest eligible Registry version.
3. Sections 6–9 create and upload an inference-ready artifact from that registered model.
4. Run Sections **12 → 14** to create the endpoint and wait for `InService`.
5. Run Section **15** to verify real inference.
6. Run Section **17** to save deployment evidence.
7. Do **not** run Section 18 cleanup until your demo/testing is complete.

If an endpoint already exists and is `InService`, it is reused by default. Set `FORCE_REDEPLOY=True` only when you intentionally want to replace it.